<a href="https://colab.research.google.com/github/daviminati/Trab_Final_IA/blob/main/notebooks/Sprint_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 5.2 Entregável 2: Avaliação do Modelo e Funções de Perda

Nesta seção, implementamos as funções para calcular a perda de entropia cruzada (*Cross-Entropy Loss*) e a **Perplexidade** para os conjuntos de dados de treinamento e validação, permitindo monitorar o aprendizado do modelo.

In [1]:
# Importações fundamentais
import torch
import torch.nn as nn
from importlib.metadata import version

print("torch version:", version("torch"))

# Função para calcular a perda (loss) de um lote (batch)
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    logits = model(input_batch)

    # O PyTorch espera logits no formato (Batch * Seq_Len, Vocab_Size)
    # e targets no formato (Batch * Seq_Len)
    loss = nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )
    return loss

# Função para calcular a perda média em todo um DataLoader
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    model.eval()
    with torch.no_grad():
        for i, (input_batch, target_batch) in enumerate(data_loader):
            if i >= num_batches:
                break
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()

    return total_loss / num_batches

torch version: 2.14.1+cpu


## 5.3 Entregável 3: Rotina de Treinamento do GPT

Implementamos a rotina principal de treinamento baseada no otimizador **AdamW**. O código itera por épocas e lotes, zerando os gradientes (`optimizer.zero_grad()`), calculando a retropropagação (`loss.backward()`) e atualizando os pesos dos tensores do modelo.

In [2]:
# Função principal de treinamento do modelo GPT
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)

            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            # Avaliação periódica durante a época
            if global_step % eval_freq == 0:
                train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
                val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

                print(f"Epoca {epoch+1} (Passo {global_step:03d}): "
                      f"Treino Loss {train_loss:.4f} | Val Loss {val_loss:.4f}")

    return train_losses, val_losses, track_tokens_seen

## 5.4 Entregável 4: Experimentos de Treinamento e Configuração de Dados

Para testar o treinamento na prática sem precisar de um dataset gigantesco, simulamos os tensores de dados de treino/validação e executamos uma sessão rápida de treinamento com o otimizador AdamW, registrando as perdas e a convergência.

In [3]:
from torch.utils.data import Dataset, DataLoader

# Dataset simulado para demonstração prática da Sprint 5
class DummyDataset(Dataset):
    def __init__(self, input_ids, target_ids):
        self.input_ids = input_ids
        self.target_ids = target_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Configuração do modelo e dados de teste
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 256, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False
}

# Importando a classe GPTModel da Sprint 4 (ou redefinindo caso necessário)
# Certifique-se de que as classes da Sprint 4 (TransformerBlock, GPTModel, etc.) já foram executadas nas células acima.

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instanciando o modelo para treino
model_gpt = GPTModel(GPT_CONFIG_124M)
model_gpt.to(device)

optimizer = torch.optim.AdamW(model_gpt.parameters(), lr=0.0004, weight_decay=0.1)

# Criando tensores fictícios de treino para validar a rotina
dummy_inputs = torch.randint(0, 50257, (16, 128))
dummy_targets = torch.randint(0, 50257, (16, 128))
train_dataset = DummyDataset(dummy_inputs, dummy_targets)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

val_dataset = DummyDataset(dummy_inputs[:4], dummy_targets[:4])
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Dispositivo de execução: {device}")
print("Iniciando rotina de testes de treinamento...")

# Executando 2 épocas de teste rápido
train_losses, val_losses, tokens_tracked = train_model_simple(
    model=model_gpt, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, device=device, num_epochs=2, eval_freq=2,
    eval_iter=1, start_context="Hello", tokenizer=None
)

NameError: name 'GPTModel' is not defined